# Meta-Memory | Agent Memory System

In [1]:
from dataclasses import dataclass, field
from typing import Dict, List
from datetime import datetime

In [2]:
# Meta-Memory System

@dataclass
class MemoryMeta:
    memory_id: str
    memory_type: str
    access_count: int = 0
    success_count: int = 0
    failure_count: int = 0
    last_accessed: datetime = field(default_factory=datetime.now)

    @property
    def reliability(self) -> float:
        total = self.success_count + self.failure_count
        return self.success_count / total if total > 0 else 0.5

    @property
    def staleness_days(self) -> float:
        return (datetime.now() - self.last_accessed).total_seconds() / 86400

class MetaMemory:
    def __init__(self):
        self.metadata: Dict[str, MemoryMeta] = {}

    def register(self, memory_id: str, memory_type: str):
        self.metadata[memory_id] = MemoryMeta(memory_id=memory_id, memory_type=memory_type)

    def record_access(self, memory_id: str, success: bool):
        if memory_id in self.metadata:
            m = self.metadata[memory_id]
            m.access_count += 1
            m.last_accessed = datetime.now()
            if success:
                m.success_count += 1
            else:
                m.failure_count += 1

    def get_unreliable(self, threshold: float = 0.5) -> List[MemoryMeta]:
        return [m for m in self.metadata.values()
                if m.reliability < threshold and (m.success_count + m.failure_count) >= 3]

    def dashboard(self) -> str:
        lines = ["META-MEMORY DASHBOARD", "=" * 40]
        for mtype in set(m.memory_type for m in self.metadata.values()):
            mems = [m for m in self.metadata.values() if m.memory_type == mtype]
            avg_rel = sum(m.reliability for m in mems) / max(len(mems), 1)
            lines.append(f"  {mtype}: {len(mems)} memories, avg reliability: {avg_rel:.0%}")
        return "\n".join(lines)

    def optimize(self):
        """Prune unreliable memories and boost high-reliability ones."""
        pruned = []
        for mid, meta in list(self.metadata.items()):
            reliability = meta.success_count / max(meta.success_count + meta.failure_count, 1)
            if reliability < 0.3 and (meta.success_count + meta.failure_count) >= 3:
                pruned.append(mid)
                del self.metadata[mid]
        if pruned:
            print(f"  Pruned {len(pruned)} unreliable memories: {pruned}")
        return pruned

In [3]:
mm = MetaMemory()
mm.register("ep-1", "episodic")
mm.register("sem-1", "semantic")
mm.register("proc-1", "procedural")
mm.record_access("ep-1", True)
mm.record_access("ep-1", True)
mm.record_access("sem-1", True)
mm.record_access("proc-1", False)
mm.record_access("proc-1", False)
mm.record_access("proc-1", False)  # 3 failures -> unreliable
print(mm.dashboard())
print(f"Unreliable: {[m.memory_id for m in mm.get_unreliable(0.6)]}")

# Adaptive behavior: warn about unreliable memory types
for m in mm.metadata.values():
    if m.reliability < 0.5 and (m.success_count + m.failure_count) >= 2:
        print(f"WARNING: {m.memory_type} memory unreliable ({m.reliability:.0%}) — consider pruning")

# Act on it: prune unreliable memories
print(f"\nBefore optimize: {list(mm.metadata.keys())}")
mm.optimize()
print(f"After optimize: {list(mm.metadata.keys())}")

META-MEMORY DASHBOARD
  procedural: 1 memories, avg reliability: 0%
  episodic: 1 memories, avg reliability: 100%
  semantic: 1 memories, avg reliability: 100%
Unreliable: ['proc-1']

Before optimize: ['ep-1', 'sem-1', 'proc-1']
  Pruned 1 unreliable memories: ['proc-1']
After optimize: ['ep-1', 'sem-1']
